# 03a — Training EfficientNetB0

**Fase 2** — Training EfficientNetB0 x 5 seed x 2 stage (freeze -> fine-tune)

**Referensi:** Proposal 3.3.6, 3.3.7

---

**Konfigurasi:**
- Stage 1: Freeze base model, LR=1e-3, 10 epoch
- Stage 2: Unfreeze 20 layer terakhir, LR=1e-4, 10 epoch
- Callbacks: EarlyStopping(p=5), ReduceLROnPlateau(f=0.5,p=5), ModelCheckpoint
- 5 seed: [42, 123, 2024, 7, 99]

**Output:**
- Checkpoint: `models/efficientnetb0_seed{n}.keras`
- History JSON: `results/history_efficientnetb0_seed{n}.json`

## Langkah 1 — Setup Environment (Clone Repo)

In [ ]:
# ================================================================
# BOOTSTRAP: Auto-detect project root + setup paths
# ================================================================
from pathlib import Path
import os, sys

def find_project_root():
    current = Path.cwd()
    for path in [current, *current.parents]:
        if (path / "requirements.txt").exists() and (path / "src").exists():
            return path
    raise RuntimeError(
        "PROJECT_ROOT TUGAS-AKHIR tidak ditemukan. "
        " Pastikan notebook dijalankan dari dalam folder repo."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Dataset: diasumsikan di-upload sebagai Kaggle Dataset privat
# Nama: dataset-cuaca-split
INPUT_DIR = Path("/kaggle/input/dataset-cuaca-split")
if not INPUT_DIR.exists():
    # Fallback: cari di PROJECT_ROOT/dataset/split (untuk development lokal)
    INPUT_DIR = PROJECT_ROOT / "dataset" / "split"

DATA_DIR    = INPUT_DIR          # dataset/split/ ada di dalam INPUT_DIR
MODELS_DIR  = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("INPUT_DIR   :", INPUT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("CWD         :", Path.cwd())
print()
assert INPUT_DIR.exists(), f"INPUT_DIR tidak ditemukan: {INPUT_DIR}"

## Langkah 2 — Import Library

In [ ]:
# ================================================================
# BOOTSTRAP: Auto-detect project root + setup paths
# ================================================================
from pathlib import Path
import os, sys

def find_project_root():
    current = Path.cwd()
    for path in [current, *current.parents]:
        if (path / "requirements.txt").exists() and (path / "src").exists():
            return path
    raise RuntimeError(
        "PROJECT_ROOT TUGAS-AKHIR tidak ditemukan. "
        " Pastikan notebook dijalankan dari dalam folder repo."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Dataset: diasumsikan di-upload sebagai Kaggle Dataset privat
# Nama: dataset-cuaca-split
INPUT_DIR = Path("/kaggle/input/dataset-cuaca-split")
if not INPUT_DIR.exists():
    # Fallback: cari di PROJECT_ROOT/dataset/split (untuk development lokal)
    INPUT_DIR = PROJECT_ROOT / "dataset" / "split"

DATA_DIR    = INPUT_DIR          # dataset/split/ ada di dalam INPUT_DIR
MODELS_DIR  = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("INPUT_DIR   :", INPUT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("CWD         :", Path.cwd())
print()
assert INPUT_DIR.exists(), f"INPUT_DIR tidak ditemukan: {INPUT_DIR}"

## Langkah 3 — Verifikasi `data_pipeline`

In [ ]:
# ================================================================
# BOOTSTRAP: Auto-detect project root + setup paths
# ================================================================
from pathlib import Path
import os, sys

def find_project_root():
    current = Path.cwd()
    for path in [current, *current.parents]:
        if (path / "requirements.txt").exists() and (path / "src").exists():
            return path
    raise RuntimeError(
        "PROJECT_ROOT TUGAS-AKHIR tidak ditemukan. "
        " Pastikan notebook dijalankan dari dalam folder repo."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Dataset: diasumsikan di-upload sebagai Kaggle Dataset privat
# Nama: dataset-cuaca-split
INPUT_DIR = Path("/kaggle/input/dataset-cuaca-split")
if not INPUT_DIR.exists():
    # Fallback: cari di PROJECT_ROOT/dataset/split (untuk development lokal)
    INPUT_DIR = PROJECT_ROOT / "dataset" / "split"

DATA_DIR    = INPUT_DIR          # dataset/split/ ada di dalam INPUT_DIR
MODELS_DIR  = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("INPUT_DIR   :", INPUT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("CWD         :", Path.cwd())
print()
assert INPUT_DIR.exists(), f"INPUT_DIR tidak ditemukan: {INPUT_DIR}"

## Langkah 4 — Load Datasets (Demo: seed=42)

In [ ]:
preprocess_eff = tf.keras.applications.efficientnet.preprocess_input

# Demo dengan seed=42
DEMO_SEED = 42

print(f'Loading datasets for seed={DEMO_SEED}...')
print('Train (augmented):')
ds_train_demo = build_tf_dataset(
    seed=DEMO_SEED,
    subset='train',
    preprocess_fn=preprocess_eff,
    augment=True,
    batch_size=32,
)

print('Val (no augmentation):')
ds_val_demo = build_tf_dataset(
    seed=DEMO_SEED,
    subset='val',
    preprocess_fn=preprocess_eff,
    augment=False,
    batch_size=32,
)

print('Test (no augmentation):')
ds_test_demo = build_tf_dataset(
    seed=DEMO_SEED,
    subset='test',
    preprocess_fn=preprocess_eff,
    augment=False,
    batch_size=32,
)

# Hitung steps
steps_train = len(list(ds_train_demo))
steps_val   = len(list(ds_val_demo))
steps_test  = len(list(ds_test_demo))

print(f'Train: {steps_train} steps, Val: {steps_val} steps, Test: {steps_test} steps')
print('Class names:', CLASS_NAMES)

## Langkah 5 — Build EfficientNetB0 Model (Stage 1: Freeze)

In [ ]:
# Bangun model Stage 1 (freeze)
print('Building EfficientNetB0 (Stage 1: freeze)...')
model_demo, unfreeze_names, total_layers = build_model(
    architecture='efficientnetb0',
    seed=DEMO_SEED,
    preprocess_fn=preprocess_eff,
)

print_model_info(model_demo)
print()
print(f'Base model total layers : {total_layers}')
print(f'Layer di-unfreeze (Stage 2): {len(unfreeze_names)} layer terakhir')
print('Sample unfreeze layer names (5 pertama):', unfreeze_names[:5])

## Langkah 6 — Stage 1: Feature Extraction (Freeze, 10 Epoch)

In [ ]:
# ================================================================
# BOOTSTRAP: Auto-detect project root + setup paths
# ================================================================
from pathlib import Path
import os, sys

def find_project_root():
    current = Path.cwd()
    for path in [current, *current.parents]:
        if (path / "requirements.txt").exists() and (path / "src").exists():
            return path
    raise RuntimeError(
        "PROJECT_ROOT TUGAS-AKHIR tidak ditemukan. "
        " Pastikan notebook dijalankan dari dalam folder repo."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Dataset: diasumsikan di-upload sebagai Kaggle Dataset privat
# Nama: dataset-cuaca-split
INPUT_DIR = Path("/kaggle/input/dataset-cuaca-split")
if not INPUT_DIR.exists():
    # Fallback: cari di PROJECT_ROOT/dataset/split (untuk development lokal)
    INPUT_DIR = PROJECT_ROOT / "dataset" / "split"

DATA_DIR    = INPUT_DIR          # dataset/split/ ada di dalam INPUT_DIR
MODELS_DIR  = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("INPUT_DIR   :", INPUT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("CWD         :", Path.cwd())
print()
assert INPUT_DIR.exists(), f"INPUT_DIR tidak ditemukan: {INPUT_DIR}"

## Langkah 7 — Stage 2: Fine-Tuning (Unfreeze, 10 Epoch)

In [ ]:
print('=' * 60)
print('STAGE 2 — Fine-Tuning (unfreeze top layers, LR=1e-4, 10 epoch)')
print('=' * 60)

# Unfreeze untuk fine-tuning
model_demo = unfreeze_for_fine_tune(model_demo)
print_model_info(model_demo)
print()
print(f'Unfreeze: {EFFICIENTNET_UNFREEZE} layer terakhir')
print(f'LR Stage 2: {STAGE2_LR}')
print(f'Epochs: {EPOCHS_STAGE2}')

callbacks_s2 = build_callbacks(checkpoint_path=final_path)

t_start = time.time()
history_s2 = model_demo.fit(
    ds_train_demo,
    validation_data=ds_val_demo,
    epochs=EPOCHS_STAGE2,
    callbacks=callbacks_s2,
    verbose=1,
)
t_stage2 = time.time() - t_start

# Muat best weights final
model_demo.load_weights(final_path)

print()
print(f'Stage 2 done in {t_stage2:.1f}s')
print(f'Best val_loss: {min(history_s2.history["val_loss"]):.4f}')
print(f'Best val_accuracy: {max(history_s2.history["val_accuracy"]):.4f}')
print(f'Final checkpoint: {final_path}')

## Langkah 8 — Visualisasi Training History

In [ ]:
# ================================================================
# BOOTSTRAP: Auto-detect project root + setup paths
# ================================================================
from pathlib import Path
import os, sys

def find_project_root():
    current = Path.cwd()
    for path in [current, *current.parents]:
        if (path / "requirements.txt").exists() and (path / "src").exists():
            return path
    raise RuntimeError(
        "PROJECT_ROOT TUGAS-AKHIR tidak ditemukan. "
        " Pastikan notebook dijalankan dari dalam folder repo."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Dataset: diasumsikan di-upload sebagai Kaggle Dataset privat
# Nama: dataset-cuaca-split
INPUT_DIR = Path("/kaggle/input/dataset-cuaca-split")
if not INPUT_DIR.exists():
    # Fallback: cari di PROJECT_ROOT/dataset/split (untuk development lokal)
    INPUT_DIR = PROJECT_ROOT / "dataset" / "split"

DATA_DIR    = INPUT_DIR          # dataset/split/ ada di dalam INPUT_DIR
MODELS_DIR  = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("INPUT_DIR   :", INPUT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("CWD         :", Path.cwd())
print()
assert INPUT_DIR.exists(), f"INPUT_DIR tidak ditemukan: {INPUT_DIR}"

## Langkah 9 — Evaluasi pada Test Set (seed=42)

In [ ]:
print('Evaluating on test set (seed=42)...')
print()

# Prediksi
y_true = []
y_pred = []
for batch_images, batch_labels in ds_test_demo:
    preds = model_demo.predict(batch_images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=-1))
    y_true.extend(batch_labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Accuracy
accuracy = np.mean(y_true == y_pred)
print(f'Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')

# Classification report
from sklearn.metrics import classification_report, confusion_matrix
print()
print('Classification Report (macro avg):')
print(classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    digits=4,
))

## Langkah 10 — Confusion Matrix (Normalized)

In [ ]:
# ================================================================
# BOOTSTRAP: Auto-detect project root + setup paths
# ================================================================
from pathlib import Path
import os, sys

def find_project_root():
    current = Path.cwd()
    for path in [current, *current.parents]:
        if (path / "requirements.txt").exists() and (path / "src").exists():
            return path
    raise RuntimeError(
        "PROJECT_ROOT TUGAS-AKHIR tidak ditemukan. "
        " Pastikan notebook dijalankan dari dalam folder repo."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Dataset: diasumsikan di-upload sebagai Kaggle Dataset privat
# Nama: dataset-cuaca-split
INPUT_DIR = Path("/kaggle/input/dataset-cuaca-split")
if not INPUT_DIR.exists():
    # Fallback: cari di PROJECT_ROOT/dataset/split (untuk development lokal)
    INPUT_DIR = PROJECT_ROOT / "dataset" / "split"

DATA_DIR    = INPUT_DIR          # dataset/split/ ada di dalam INPUT_DIR
MODELS_DIR  = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("INPUT_DIR   :", INPUT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("CWD         :", Path.cwd())
print()
assert INPUT_DIR.exists(), f"INPUT_DIR tidak ditemukan: {INPUT_DIR}"

## Langkah 11 — Simpan History & Metrik ke JSON

In [ ]:
# ================================================================
# BOOTSTRAP: Auto-detect project root + setup paths
# ================================================================
from pathlib import Path
import os, sys

def find_project_root():
    current = Path.cwd()
    for path in [current, *current.parents]:
        if (path / "requirements.txt").exists() and (path / "src").exists():
            return path
    raise RuntimeError(
        "PROJECT_ROOT TUGAS-AKHIR tidak ditemukan. "
        " Pastikan notebook dijalankan dari dalam folder repo."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Dataset: diasumsikan di-upload sebagai Kaggle Dataset privat
# Nama: dataset-cuaca-split
INPUT_DIR = Path("/kaggle/input/dataset-cuaca-split")
if not INPUT_DIR.exists():
    # Fallback: cari di PROJECT_ROOT/dataset/split (untuk development lokal)
    INPUT_DIR = PROJECT_ROOT / "dataset" / "split"

DATA_DIR    = INPUT_DIR          # dataset/split/ ada di dalam INPUT_DIR
MODELS_DIR  = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("INPUT_DIR   :", INPUT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("CWD         :", Path.cwd())
print()
assert INPUT_DIR.exists(), f"INPUT_DIR tidak ditemukan: {INPUT_DIR}"

## Langkah 12 — Full Pipeline: Semua 5 Seed (Sequential)

In [ ]:
# ================================================================
# FULL PIPELINE — 5 seed × EfficientNetB0 × 2-stage training
# ================================================================
# JANGAN JALANKAN CELL INI SEKALIGUS DENGAN CELL 6-11
# Cell 6-11 adalah DEMO (seed=42). Ini pipeline untuk SEMUA seed.
# ================================================================

import gc

# Hapus model demo dari memory
del model_demo
del ds_train_demo, ds_val_demo, ds_test_demo
del history_s1, history_s2
gc.collect()
tf.keras.backend.clear_session()

print('=' * 60)
print('FULL PIPELINE — EfficientNetB0 x 5 seeds')
print('=' * 60)

t_pipeline_start = time.time()
all_results = {}

for seed in SEEDS:
    print()
    print('=' * 60)
    print(f'SEED {seed} / {SEEDS}')
    print('=' * 60)

    t_seed_start = time.time()

    # Load datasets
    print('Loading datasets...')
    ds_train = build_tf_dataset(
        seed=seed, subset='train',
        preprocess_fn=preprocess_eff,
        augment=True, batch_size=32,
    )
    ds_val = build_tf_dataset(
        seed=seed, subset='val',
        preprocess_fn=preprocess_eff,
        augment=False, batch_size=32,
    )

    # Train
    ckpt_path = str(MODELS_DIR / f'efficientnetb0_seed{seed}.keras')
    result = train_two_stage(
        architecture='efficientnetb0',
        seed=seed,
        preprocess_fn=preprocess_eff,
        ds_train=ds_train,
        ds_val=ds_val,
        checkpoint_path=ckpt_path,
        verbose=1,
    )
    all_results[seed] = result

    # Cleanup
    del result['model'], ds_train, ds_val
    gc.collect()
    tf.keras.backend.clear_session()

    t_seed = time.time() - t_seed_start
    print(f'Seed {seed} done in {t_seed:.1f}s')

t_total = time.time() - t_pipeline_start
print()
print('=' * 60)
print('ALL 5 SEEDS COMPLETE')
print(f'Total time: {t_total:.1f}s ({t_total/60:.1f} minutes)')
print('=' * 60)

## Langkah 13 — Simpan Semua Metrik (5 Seed)

In [ ]:
print('Saving all metrics...')
print()

all_metrics = []

for seed in SEEDS:
    mpath = results_dir / f'metrics_efficientnetb0_seed{seed}.json'
    if mpath.exists():
        with open(mpath) as f:
            m = json.load(f)
        all_metrics.append(m)
        print(f'Seed {seed}: acc={m["test_accuracy"]:.4f} '
              f'f1={m["test_f1_macro"]:.4f}')

# Summary
if all_metrics:
    df = pd.DataFrame(all_metrics)
    print()
    print('=== SUMMARY ===')
    print(df[['seed','test_accuracy','test_precision_macro',
              'test_recall_macro','test_f1_macro']].to_string(index=False))
    print()
    print(f'Mean accuracy : {df["test_accuracy"].mean():.4f} +/- {df["test_accuracy"].std():.4f}')
    print(f'Mean F1 macro: {df["test_f1_macro"].mean():.4f} +/- {df["test_f1_macro"].std():.4f}')

    # Save summary
    summary_path = results_dir / 'metrics_efficientnetb0_summary.csv'
    df.to_csv(summary_path, index=False)
    print()
    print('Summary saved:', summary_path)